For $\alpha$ runs:

- Load the sophronia/dorothea reconstruced data by IC.
- Apply the selecting criteria of your preference.
- Store all the dataframes for future analysis.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis')

from libs import crudo

import os
import pandas as pd
import pickle

%load_ext autoreload
%autoreload 2

# Runs Information

In [2]:
# α runs!
RUNS_INFO = [    
                # --- Cold-Getter: Jan 17th --- #
                # {"run_number": 14714, "duration": 64574, "OK": 1909287, "LOST": 1728359, "real_rate": 56.333},
                # # {"run_number": 14715, "duration": 84365, "OK": 2469062, "LOST": 2239303, "real_rate": 55.809},      # Waveforms in magnetic tape, not processed
                {"run_number": 14716, "duration": 17036, "OK": 495769 , "LOST": 451306 , "real_rate": 55.592},
                # {"run_number": 14720, "duration": 48518, "OK": 1432110, "LOST": 1294941, "real_rate": 56.207},      # Estimated from fit
                # {"run_number": 14733, "duration": 53881, "OK": 1587637, "LOST": 1429629, "real_rate": 55.998},
                # {"run_number": 14735, "duration": 84987, "OK": 2508569, "LOST": 2267982, "real_rate": 56.203},
                # {"run_number": 14737, "duration": 72705, "OK": 2153786, "LOST": 1960347, "real_rate": 56.586},
                # {"run_number": 14739, "duration": 87138, "OK": 2576630, "LOST": 2334343, "real_rate": 56.358},
                # {"run_number": 14741, "duration": 87755, "OK": 2592615, "LOST": 2349583, "real_rate": 56.318},
                # {"run_number": 14743, "duration": 82332, "OK": 2437984, "LOST": 2220643, "real_rate": 56.583},
                # {"run_number": 14745, "duration": 60990, "OK": 1803415, "LOST": 1636317, "real_rate": 56.398},

                # # --- Hot-Getter: Jan 27th --- #
                # {"run_number": 14753, "duration": 86308, "OK": 2188547, "LOST": 1532351, "real_rate": 43.111},
                # {"run_number": 14765, "duration": 55978, "OK": 1259832, "LOST": 735202 , "real_rate": 35.639},
                # {"run_number": 14776, "duration": 47397, "OK": 957081 , "LOST": 481810 , "real_rate": 30.358},        # Data removed
                # {"run_number": 14780, "duration": 88503, "OK": 1600797, "LOST": 682246 , "real_rate": 25.796},
                # # Feb 2025
                # {"run_number": 14782, "duration": 85654, "OK": 1382153, "LOST": 506234 , "real_rate": 22.046},
                # {"run_number": 14784, "duration": 57290, "OK": 842541 , "LOST": 273581 , "real_rate": 19.481},
                # {"run_number": 14789, "duration": 74148, "OK": 976360 , "LOST": 277504 , "real_rate": 16.910},
                # # Zero Suppression
                # {"run_number": 14803, "duration": 71721, "OK": 933727 , "LOST": 104677 , "real_rate": 14.478},
                # {"run_number": 14804, "duration": 65116, "OK": 680552 , "LOST": 156995 , "real_rate": 12.862},
                # # NO Zero Suppression
                # {"run_number": 14811, "duration": 84651, "OK": 782816 , "LOST": 147773 , "real_rate": 10.993},
                # {"run_number": 14814, "duration": 6344 , "OK": 55679  , "LOST": 10155  , "real_rate": 10.377},
                # {"run_number": 14815, "duration": 86232, "OK": 717786 , "LOST": 121527 , "real_rate": 9.733 },
                # {"run_number": 14816, "duration": 86580, "OK": 659265 , "LOST": 101374 , "real_rate": 8.785 },
                # {"run_number": 14817, "duration": 49662, "OK": 352203 , "LOST": 50474  , "real_rate": 8.108 },
                # {"run_number": 14828, "duration": 53609, "OK": 300827 , "LOST": 100405 , "real_rate": 7.484 },
                # {"run_number": 14829, "duration": 73042, "OK": 387003 , "LOST": 121249 , "real_rate": 6.958 },
                # # Low Radon
                # {"run_number": 14834, "duration": 5779 , "OK": 32864  , "LOST": 3887   , "real_rate": 6.359 },
                # {"run_number": 14835, "duration": 11713, "OK": 66515  , "LOST": 7358   , "real_rate": 6.306 },
                # {"run_number": 14837, "duration": 55751, "OK": 294175 , "LOST": 31034  , "real_rate": 5.833 },
                # {"run_number": 14838, "duration": 87854, "OK": 431766 , "LOST": 42219  , "real_rate": 5.395 },
                # {"run_number": 14839, "duration": 84881, "OK": 401004 , "LOST": 37747  , "real_rate": 5.169 },
                # {"run_number": 14840, "duration": 56174, "OK": 260266 , "LOST": 23627  , "real_rate": 5.054 }  
            ]

In [13]:
SELECTION_CRITERIA = {  'Xe': (lambda x:  (x['nS1'].sum() == 1)
                                        & (x['nS2'].sum() == 1)
                                        & (0     <= x['S1w'].sum() <= 1.25e3)  # In [ns]
                                        & (150e3 <= x['S1t'].sum()          )  # In [ns]
                                        & (0     <= x['S2w'].sum() <= 250   )  # In [μs]
                                        & (         x['S2t'].sum() <= 1650e3)  # In [ns]
                                        # 214Po-like events: this cut is waveform-based, none correction needed
                                        & (x['S1h'].sum() >= 0.17 * x['S1e'].sum() - 56)    # Reject (>=) or select (<)
                                ),
                      }

# Reconstructed Data

In [5]:
RECO_DICT = {}

for run in RUNS_INFO:

    run_id = run['run_number']
    # Load and assign the data
    RECO_DICT[run_id] = crudo.dm.load_run_data(run_id, key='/DST/Events', trigger2=False)

/DST/Events: Run 14714 successfully loaded with data shape: (295774, 26)
/DST/Events: Run 14716 successfully loaded with data shape: (147883, 26)
/DST/Events: Run 14720 successfully loaded with data shape: (427675, 26)
/DST/Events: Run 14733 successfully loaded with data shape: (246994, 26)
/DST/Events: Run 14735 successfully loaded with data shape: (380009, 26)
/DST/Events: Run 14737 successfully loaded with data shape: (337148, 26)
/DST/Events: Run 14739 successfully loaded with data shape: (403582, 26)
/DST/Events: Run 14741 successfully loaded with data shape: (405333, 26)
/DST/Events: Run 14743 successfully loaded with data shape: (381001, 26)
/DST/Events: Run 14745 successfully loaded with data shape: (280653, 26)
/DST/Events: Run 14753 successfully loaded with data shape: (302638, 26)
/DST/Events: Run 14765 successfully loaded with data shape: (244159, 26)
/DST/Events: Run 14776 successfully loaded with data shape: (26, 26)
/DST/Events: Run 14780 successfully loaded with data sh

# Selected Data

In [14]:
%%time
SEL_DICT = {key : {} for key in RECO_DICT.keys()}

for key in RECO_DICT.keys():
    
    print(f"Run {key}:")
    # Dorothea-level
    doro_df = crudo.dm.filter_run_data(RECO_DICT[key], SELECTION_CRITERIA['Xe'])
    SEL_DICT[key] = doro_df

Run 14714:
Filtered successfully. Data shape: (70357, 26)
Run 14716:
Filtered successfully. Data shape: (10949, 26)
Run 14720:
Filtered successfully. Data shape: (30834, 26)
Run 14733:
Filtered successfully. Data shape: (56634, 26)
Run 14735:
Filtered successfully. Data shape: (90552, 26)
Run 14737:


KeyboardInterrupt: 

In [12]:
%%time
SEL_DATA = {run["run_number"]: crudo.dm.filter_run_data(run, RECO_DATA, selection_criteria)[run["run_number"]] for run in RUNS_INFO}

Run 14714 filtered successfully. Data shape: (70293, 26)
Run 14716 filtered successfully. Data shape: (10933, 26)
Run 14720 filtered successfully. Data shape: (30787, 26)
Run 14733 filtered successfully. Data shape: (56598, 26)
Run 14735 filtered successfully. Data shape: (90465, 26)
Run 14737 filtered successfully. Data shape: (80106, 26)
Run 14739 filtered successfully. Data shape: (96087, 26)
Run 14741 filtered successfully. Data shape: (96978, 26)
Run 14743 filtered successfully. Data shape: (90420, 26)
Run 14745 filtered successfully. Data shape: (66399, 26)
Run 14753 filtered successfully. Data shape: (79061, 26)
Run 14765 filtered successfully. Data shape: (27755, 26)
Run 14776 filtered successfully. Data shape: (3, 26)
Run 14780 filtered successfully. Data shape: (39659, 26)
Run 14782 filtered successfully. Data shape: (55110, 26)
Run 14784 filtered successfully. Data shape: (32390, 26)
Run 14789 filtered successfully. Data shape: (32979, 26)
Run 14803 filtered successfully. Da

# Dataframes Storage

In [ ]:
# Dataframe name
df_name = 's2w_no_cuts'   # 'Rn_background', 's2w_no_cuts'

# Choose which data to store: SEL_DICT or RECO_DICT
data = SEL_DICT

# Store the selected data
with open(f"/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/{df_name}.pkl", "wb") as file:
    pickle.dump(data, file)
    
print('Listo mi pana, péguese un análisis sabroso!')

Listo mi pana, péguese un análisis sabroso!


#### Or do you wanna merge two different dataframes into one?
<span style="color:red">OUTDATED</span>

In [ ]:
merged_data = crudo.dm.merge_dfs( file1="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/temp_runs.pkl", 
                                  file2="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/extra_runs.pkl", 
                                  output_file="/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Rn_analysis/pkl/alpha_runs.pkl" )

Stored merged data in: /data_extra2/ccortesp/NEXT-100/Xe_cmmssnng/data/alpha_runs.pkl
